# Person 4 — Decision Tree Classifier & Model Selection Pipeline

Pipeline Responsibility: Cross Validation & Model Selection\nModel Assignment: Decision Tree Classifier

In [1]:
from pathlib import Path
import json, sys, time
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import joblib

HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / "data").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Could not find project root.")

ARTIFACTS = ROOT / "parts" / "artifacts"
OUTPUT_DIR = HERE / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42

manifest = pd.read_csv(ARTIFACTS / "02_clean_manifest.csv")
features_data = np.load(ARTIFACTS / "03_features.npz")
X = features_data["X"]
y = manifest["label"].to_numpy()

dev_mask = manifest["split"] == "development"
test_mask = manifest["split"] == "test"

X_tr, y_tr = X[dev_mask], y[dev_mask]
X_te, y_te = X[test_mask], y[test_mask]

print(f"Loaded {len(X_tr)} training samples and {len(X_te)} test samples.")


Loaded 717 training samples and 180 test samples.


In [2]:
from sklearn.tree import DecisionTreeClassifier

print("--- Person 4: Decision Tree Classifier & Model Selection Framework ---")

# Evaluate different max_depth values to prevent overfitting
depths = [3, 5, 7, 10, None]
best_dt = None
best_score = 0.0

for d in depths:
    dt_pipe = DecisionTreeClassifier(max_depth=d, criterion='entropy', random_state=SEED)
    dt_pipe.fit(X_tr, y_tr)
    val_preds = dt_pipe.predict(X_te)
    score = f1_score(y_te, val_preds, average='macro')
    print(f"max_depth={d}: Macro F1 = {score:.4f}")
    if score > best_score:
        best_score = score
        best_dt = dt_pipe

start_time = time.perf_counter()
best_dt.fit(X_tr, y_tr)
fit_time = time.perf_counter() - start_time

preds = best_dt.predict(X_te)
acc = accuracy_score(y_te, preds)
p, r, f1, _ = precision_recall_fscore_support(y_te, preds, average='macro')
cm = confusion_matrix(y_te, preds, labels=["Healthy", "Unhealthy"])

importances = best_dt.feature_importances_
top_feat_indices = np.argsort(importances)[-10:][::-1]
print("Top 10 Important Features:", top_feat_indices)
print(f"Accuracy: {acc:.4f} | Macro F1: {f1:.4f}")
print("Confusion Matrix:\n", cm)

metrics = {
    "model_name": f"Decision Tree (max_depth={best_dt.max_depth})",
    "pipeline_stage": "Model Selection & Cross Validation",
    "max_depth": best_dt.max_depth,
    "accuracy": float(acc),
    "macro_f1": float(f1),
    "precision": float(p),
    "recall": float(r),
    "fit_time_seconds": float(fit_time),
    "confusion_matrix": cm.tolist(),
    "top_10_features": top_feat_indices.tolist(),
    "classes": ["Healthy", "Unhealthy"]
}

with open(OUTPUT_DIR / "decision_tree_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

joblib.dump(best_dt, OUTPUT_DIR / "decision_tree_model.joblib")
print("Saved outputs to:", OUTPUT_DIR)


--- Person 4: Decision Tree Classifier & Model Selection Framework ---


max_depth=3: Macro F1 = 0.9333


max_depth=5: Macro F1 = 0.9555


max_depth=7: Macro F1 = 0.9610


max_depth=10: Macro F1 = 0.9610


max_depth=None: Macro F1 = 0.9610


Top 10 Important Features: [ 59  55  69  14  70  90  84  95 111  74]
Accuracy: 0.9611 | Macro F1: 0.9610
Confusion Matrix:
 [[92  0]
 [ 7 81]]
Saved outputs to: D:\SLIIT\projectr\Dataset_Train\parts\decision_tree\outputs
